In [40]:
from qdrant_client import QdrantClient
import ollama
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.models import Filter, FieldCondition, MatchValue
from pypdf import PdfReader
import uuid
import re
import os

In [3]:
resp = ollama.embed(
    model = "qwen3-embedding",
    input = "Fala meu amigo como que vc está hoje?"
)

print(resp.embeddings)

[[0.04237265, -0.0015366419, -0.0068038297, -0.035773534, 0.009006815, -0.029391702, -0.013476182, -0.004761189, 0.00018382703, 0.019741291, -0.04699377, -0.014423593, 0.048244156, -0.0055936943, -0.0014011506, 0.007946659, -0.020242065, -0.0007238203, 0.021085778, 0.036003493, -0.02489101, 0.0014868325, 0.037075493, -0.030103285, 0.011103969, 0.038395345, -0.011748965, -0.035003144, 0.003422189, 0.059896898, -0.023976795, 0.019154338, 0.019403828, 0.007908501, -0.028385645, 0.013224156, 0.003852684, 0.016167253, 0.026564917, -0.008627827, -0.00038736654, 0.0057335244, -0.002706411, 0.0052263224, -0.030481646, -0.013707698, -0.00779094, -0.0038460668, 0.012811546, 0.009209637, 0.021598829, -0.028023794, -0.006521927, -0.015784353, -0.014637774, -0.0017133461, -0.017217526, -0.01377124, -0.030020649, 0.027701408, -0.04007825, 0.006939779, 0.018547565, -0.01569383, 0.007777791, -0.007585887, 0.0007452214, 0.0025116522, 0.021480775, 0.015994005, 0.0027845982, -0.02214557, -0.009549866, 0.

In [ ]:
client = QdrantClient("localhost", port=6333)

client.sear

In [31]:
client.delete_collection(collection_name="figas_teste")

True

In [33]:
collection_name = "figas_teste"

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config = VectorParams(size=4096, distance=Distance.COSINE)
    )
    
    print(f"Collection '{collection_name}' created.")    


In [20]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return text

def load_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    
    previous_tail = ""
    
    for i, page in enumerate(reader.pages):
        raw_text = page.extract_text()
        
        if not raw_text: continue 
        
        current_text = clean_text(raw_text)
        full_text_to_embed = previous_tail + " " + current_text
        
        resp = ollama.embed(
            model = "qwen3-embedding",
            input = full_text_to_embed
        )
        
        embedding = resp.embeddings[0]
        
        point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{pdf_path}_{i}"))
        
        client.upsert(
            collection_name=collection_name,
            points=[
                PointStruct(
                    id=point_id,
                    vector=embedding,
                    payload={
                            "text": full_text_to_embed, 
                            "source": pdf_path.split("/")[-1],
                            "page": i
                    }
                )
            ]
        )
        
    print(f"PDF '{pdf_path}' loaded into Qdrant. FIGAS!")
    


In [21]:
pdf_path = "/home/rafael/livros/finance" #/home/rafael/livros/finance
book_name = "'Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf'"

test = "/home/rafael/livros/finance/Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf"

load_pdf(test)

PDF '/home/rafael/livros/finance/Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf' loaded into Qdrant. FIGAS!


In [ ]:
system_prompt = """You are a high-level academic research assistant specialized in Quantitative Finance and Stochastic Calculus. 
Your primary goal is to provide rigorous, technically accurate answers based EXCLUSIVELY on the provided context.

GUIDELINES:
1. TRUTH-GROUNDING: Only use information from the retrieved context. If the context does not contain the answer, explicitly state: "I did not find specific information regarding this in the loaded documents."
2. MATHEMATICAL RIGOR: When the context includes formulas or theorems, explain them using precise notation. Maintain the logical flow of proofs if present.
3. TERMINOLOGY: Use standard industry and academic terminology (e.g., Martingales, Ito's Lemma, Sigma-algebras).
4. STRUCTURE: Use LaTeX for any mathematical expression and organize your response with clear headings or bullet points.

Respond in the language of the user's question, but maintain this internal logical framework."""


def search_figas(pergunta, system_prompt, top_k=3):
    
    resp_emb = ollama.embed(
        model = "qwen3-embedding",
        input = pergunta
    )
    
    query_vector = resp_emb.embeddings[0]

    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True
    ).points
    
    contexto_formatado = ""
    
    for hit in search_result:
        contexto_formatado += f"\n--- Trecho (Fonte: {hit.payload['source']}, Pág: {hit.payload['page']}) ---\n"
        contexto_formatado += hit.payload['text'] + "\n"
        
    user_content = f"""CONTEXT RECOVERED:
    {contexto_formatado}
    
    QUESTION: {pergunta}
    
    ANSWER THE QUESTION BASED ONLY ON THE CONTEXT ABOVE. If the context contains mathematical formulas, explain them in detail.
    """
    
    final_resp = ollama.chat(
        model="qwen3.5:4b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]
    )
    
    return final_resp.message.content, search_result

In [30]:
pergunta_teste = "What are the conditions for a process to be considered a Martingale with respect to a filtration Fn​ according to the text?"
resposta, fontes = search_figas(pergunta_teste, system_prompt)

print("### RESULTADO DA PESQUISA ###")
print(resposta)

### RESULTADO DA PESQUISA ###
Based on the provided context from *Stochastic Calculus for Finance I* (specifically snippet 1 and 2), the definition and conditions for **Martingales** (Section 2.4) are as follows:

According to the decipherable fragments in the text, the "Ingredients for Martingales" consist of specific conditions regarding a process $\{M_k\}$ and a filtration $\{F_k\}$.

### 1. The Filtration (Condition A)
The text contains the fragment `F /0 /#1AF /1 /`, which corresponds to the definition of the filtration (or $\sigma$-algebras) $\{F_k\}$:
$$ F_0 \subset F_1 \subset F_2 \subset \dots $$
*(Note: The text fragments `/#0C` and `/#15` typically represent $\sigma$ in this OCR context, implying these are increasing $\sigma$-algebras).*

### 2. Conditions for Martingales
The text explicitly lists "Ingredients for Martingales" and refers to `Condition 1` and `Condition 2`. The decipherable mathematical condition found is:
*   **Condition 1 (Adaptation):** The random variable

In [43]:
DOMAIN_KEYWORDS = {
    "finance":   ["finance", "stochastic", "options", "futures", "trading", "time series", "quantum finance"],
    "math":      ["calculus", "differential", "numerical", "partial", "discrete", "finite difference"],
    "ml":        ["reinforcement", "deep learning", "rl_book", "rl_2_book"],
    "java":      ["spring", "security"],
    "go":        ["go in practice"],
    "api":       ["api design"],
}

In [44]:

def detect_domain(filename: str) -> str:
    name = filename.lower()
    for domain, keywords in DOMAIN_KEYWORDS.items():
        if any(kw in name for kw in keywords):
            return domain
    return "general"

# ── Payload ──────────────────────────────────────────────────────────────────
def make_payload(chunk: str, source: str, type_text: str, domain: str) -> dict:
    return {
        "text": chunk,
        "source": source,
        "type": type_text,
        "domain": domain,
    }

# ── Text cleaning ─────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    return text.strip()

# ── Chunking ──────────────────────────────────────────────────────────────────
def chunk_text(text: str, chunk_size: int = 400, overlap: int = 80) -> list[str]:
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.strip()) > 50:   # skip near-empty chunks
            chunks.append(chunk)
    return chunks

def chunk_markdown(text: str) -> list[str]:
    sections = re.split(r"\n#{1,3} ", text)
    return [s.strip()[:2000] for s in sections if len(s.strip()) > 50]

# ── PDF loader ────────────────────────────────────────────────────────────────
def load_pdf_chunked(pdf_path: str, collection_name: str):
    filename = os.path.basename(pdf_path)
    domain = detect_domain(filename)

    try:
        reader = PdfReader(pdf_path)
        full_text = ""
        for page in reader.pages:
            raw = page.extract_text()
            if raw:
                full_text += clean_text(raw) + " "

        if not full_text.strip():
            print(f"  [SKIP] No text extracted: {filename}")
            return

        chunks = chunk_text(full_text)
        points = []

        for i, chunk in enumerate(chunks):
            resp = ollama.embed(model="qwen3-embedding", input=chunk)
            embedding = resp.embeddings[0]
            point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{pdf_path}_{i}"))

            points.append(PointStruct(
                id=point_id,
                vector=embedding,
                payload=make_payload(
                    chunk=chunk,
                    source=filename,
                    type_text="book",
                    domain=domain,
                )
            ))

        # batch upsert
        client.upsert(collection_name=collection_name, points=points)
        print(f"  [OK] {filename} → {len(chunks)} chunks | domain: {domain}")

    except Exception as e:
        print(f"  [ERROR] {filename}: {e}")   # don't abort the whole loop

# ── Markdown/docs loader ──────────────────────────────────────────────────────
def load_markdown(file_path: str, collection_name: str, domain: str = None):
    filename = os.path.basename(file_path)
    domain = domain or detect_domain(filename)

    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        chunks = chunk_markdown(text)
        points = []

        for i, chunk in enumerate(chunks):
            resp = ollama.embed(model="qwen3-embedding", input=chunk)
            embedding = resp.embeddings[0]

            points.append(PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding,
                payload=make_payload(
                    chunk=chunk,
                    source=filename,
                    type_text="documentation",
                    domain=domain,
                )
            ))

        client.upsert(collection_name=collection_name, points=points)
        print(f"  [OK] {filename} → {len(chunks)} chunks | domain: {domain}")

    except Exception as e:
        print(f"  [ERROR] {filename}: {e}")

# ── Ingest a full directory ───────────────────────────────────────────────────
def ingest_directory(directory: str, collection_name: str, domain: str = None):
    files = [f for f in os.listdir(directory)]
    pdfs = [f for f in files if f.endswith(".pdf")]
    mds  = [f for f in files if f.endswith((".md", ".rst", ".txt"))]

    print(f"\nIngesting '{directory}': {len(pdfs)} PDFs, {len(mds)} docs")

    for filename in pdfs:
        load_pdf_chunked(os.path.join(directory, filename), collection_name)

    for filename in mds:
        load_markdown(os.path.join(directory, filename), collection_name, domain)

In [45]:
ingest_directory("/home/rafael/livros", collection_name)


Ingesting '/home/rafael/livros': 24 PDFs, 0 docs
  [OK] Patterns of Distributed Systems (Addison-Wesley Signature -- Unmesh Joshi -- Addison-Wesley Signature Series (Fowler), 1, 2023 -- Addison-Wesley -- 9780138221980 -- d3099f1326c6a537679eec5c1e20fb8f -- Anna’s Archive.pdf → 257 chunks | domain: general
  [OK] Quantum Finance _ Intelligent Forecast and Trading Systems -- Raymond S_ T_ Lee -- 1st ed_ 2020, Singapore, 2020 -- Springer Singapore _ Imprint _ -- 9789813297951 -- 60ac9acb93d0bdfed057a8bbbcbb88f4 -- Anna’s Archive.pdf → 332 chunks | domain: finance
  [OK] A First Course in Stochastic Calculus (Pure and Applied -- Louis-Pierre Arguin -- Volume 69, Number 3, March 2022, 1, 2021 -- American Mathematical -- 9781470464882 -- 8f94594c459b525190295ea747f7bbcd -- Anna’s Archive.pdf → 328 chunks | domain: finance
  [ERROR] Analysis of Financial Time Series, Third Edition (Wiley -- Ruey S_ Tsay -- Wiley Series in Probability and Statistics, 2010 -- 2c06dca2a6bb2dbda257a0db84474cc5 -

In [74]:


def search_figas(pergunta, system_prompt, top_k=5, domain=None):
    resp_emb = ollama.embed(model="qwen3-embedding", input=pergunta)
    query_vector = resp_emb.embeddings[0]

    query_filter = None
    if domain:
        query_filter = Filter(must=[
            FieldCondition(key="domain", match=MatchValue(value=domain))
        ])

    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True,
        score_threshold=0.42,
        query_filter=query_filter
    ).points

    if not search_result:
        return "No relevant context found in the loaded documents.", []

    contexto_formatado = ""
    for hit in search_result:
        source = hit.payload.get("source", "unknown")
        domain_tag = hit.payload.get("domain", "?")
        score = hit.score
        contexto_formatado += f"\n--- Source: {source} | Domain: {domain_tag} | Score: {score:.2f} ---\n"
        contexto_formatado += hit.payload["text"] + "\n"

    user_content = f"""CONTEXT RECOVERED:
{contexto_formatado}

QUESTION: {pergunta}

ANSWER THE QUESTION BASED ONLY ON THE CONTEXT ABOVE. If the context contains mathematical formulas, explain them in detail.
"""

    final_resp = ollama.chat(
        model="qwen3:14b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]
    )

    return final_resp.message.content, search_result

In [51]:
pergunta_teste = "What are the conditions for a process to be considered a Martingale with respect to a filtration Fn​ according to the text?"
resposta, fontes = search_figas(pergunta_teste, system_prompt)

print("### RESULTADO DA PESQUISA ###")
print(resposta)

### RESULTADO DA PESQUISA ###
To determine whether a process $\{M_t\}_{t \geq 0}$ is a **martingale with respect to a filtration $\{\mathcal{F}_t\}_{t \geq 0}$**, the context provides the following **three conditions**:

---

### **1. Adaptedness**
- **Condition**: For all $t \geq 0$, the random variable $M_t$ must be **$\mathcal{F}_t$-measurable**.
- **Interpretation**: The process $\{M_t\}$ must be **adapted to the filtration $\{\mathcal{F}_t\}$**, meaning that the value of $M_t$ depends only on the information available up to time $t$ (i.e., the $\sigma$-algebra $\mathcal{F}_t$).

---

### **2. Integrability**
- **Condition**: For all $t \geq 0$, the **expected absolute value** of $M_t$ is finite:
  $$
  \mathbb{E}[|M_t|] < \infty.
  $$
- **Interpretation**: This ensures that the **conditional expectation** $\mathbb{E}[M_t \mid \mathcal{F}_s]$ is well-defined for all $s \leq t$.

---

### **3. Martingale Property**
- **Condition**: For all $s \leq t$, the **conditional expectation**

In [62]:
from IPython.display import Markdown, display

print(type(resposta))

def display_figas(pergunta_teste, system_prompt):
    
    resposta, fontes = search_figas(pergunta_teste, system_prompt)
    
    display(Markdown(resposta))
    
    print("\n--- Sources ---")
    for hit in fontes:
        print(f"  · {hit.payload.get('source')} | domain: {hit.payload.get('domain')} | score: {hit.score:.2f}")
        


<class 'str'>


In [65]:
system_prompt = """You are an elite academic and engineering assistant with deep expertise across:
- Quantitative Finance & Stochastic Calculus
- Software Engineering (Python, Go, Java/Spring, API Design, Distributed Systems)
- Machine Learning & Reinforcement Learning
- Applied Mathematics (PDEs, ODEs, Numerical Methods, Discrete Structures)

Your answers are grounded EXCLUSIVELY in the retrieved context provided to you.

═══════════════════════════════════════
CORE RULES
═══════════════════════════════════════

1. TRUTH-GROUNDING
   - Use ONLY information from the retrieved context.
   - If the answer is not in the context, say exactly:
     "I did not find specific information regarding this in the loaded documents."
   - Never infer, hallucinate, or supplement with outside knowledge.

2. MATH & THEORY
   - Render ALL mathematical expressions in LaTeX:
     · Inline: $expression$
     · Block:  $$expression$$
   - Preserve proof structure and logical flow when present.
   - Define every symbol on first use.
   - Use standard terminology: Martingales, Itô's Lemma, σ-algebras,
     Filtrations, Risk-Neutral Measure, etc.

3. CODE
   - Always specify the language in fenced code blocks:
``````python
`````go
````java
   - Explain what the code does BEFORE showing it.
   - If the context shows a specific implementation pattern, follow it exactly.
   - Highlight key lines with inline comments.

4. STRUCTURE
   - Use clear markdown headings (##, ###).
   - Prefer structured answers: definitions → theory → examples → code.
   - Use bullet points for lists, numbered steps for sequences.
   - End complex answers with a ## Summary section.

5. CROSS-DOMAIN ANSWERS
   - If the question touches both math and code (e.g., implementing a 
     stochastic model), address the theory first, then the implementation.
   - Draw explicit connections between the mathematical concept and its
     code representation.

═══════════════════════════════════════
OUTPUT FORMAT
═══════════════════════════════════════

## [Topic]

### Definition / Concept
...

### Mathematical Formulation
$$...$$

### Implementation
```language
...
```

### Summary
...

Respond in the language of the user's question.
"""


In [73]:
pergunta = "Explain to me how can i'm use the european call option to precify a option?" 

display_figas(pergunta_teste=pergunta, system_prompt=system_prompt)

To price a European call option using the context provided, follow these steps based on **risk-neutral valuation** and **arbitrage pricing theory**:

---

### **1. Understand the Payoff of a European Call Option**
A European call option gives the holder the right to buy an underlying asset at a predetermined **strike price (K)** at **expiration (T)**. Its payoff at expiration is:
$$
V_T = \max(S_T - K, 0)
$$
Where:
- $ S_T $ is the price of the underlying asset at expiration.
- $ K $ is the strike price.
- $ V_T $ is the payoff of the option at expiration.

This payoff depends on the outcome of a stochastic process (e.g., coin tosses in the binomial model) that determines whether the asset price goes up ($ S_T = S_0 \cdot u $) or down ($ S_T = S_0 \cdot d $).

---

### **2. Define Risk-Neutral Probabilities**
In the binomial model, the **risk-neutral probabilities** ($ \tilde{p} $ and $ \tilde{q} $) are calculated to ensure **no arbitrage**. These probabilities are **not** the real-world probabilities of up/down movements but are derived from the risk-free rate ($ r $) and the possible asset price movements ($ u $ and $ d $):

$$
\tilde{p} = \frac{(1 + r) - d}{u - d}, \quad \tilde{q} = 1 - \tilde{p}
$$

Where:
- $ u > 1 $: Up factor (e.g., 1.2).
- $ d < 1 $: Down factor (e.g., 0.8).
- $ r $: Risk-free interest rate (e.g., 5% or 0.05).

These probabilities ensure that the expected return of the underlying asset under the risk-neutral measure equals the risk-free rate.

---

### **3. Calculate the Expected Payoff at Expiration**
Using the risk-neutral probabilities, compute the **expected payoff** of the option at expiration:

$$
\text{Expected Payoff} = \tilde{p} \cdot V_T(H) + \tilde{q} \cdot V_T(T)
$$

Where:
- $ V_T(H) = \max(S_0 \cdot u - K, 0) $: Payoff if the asset price goes up.
- $ V_T(T) = \max(S_0 \cdot d - K, 0) $: Payoff if the asset price goes down.

---

### **4. Discount the Expected Payoff to Present Value**
To find the **current price** of the option ($ V_0 $), discount the expected payoff at the risk-free rate ($ r $):

$$
V_0 = \frac{\tilde{p} \cdot V_T(H) + \tilde{q} \cdot V_T(T)}{1 + r}
$$

This formula ensures the option is priced to eliminate arbitrage opportunities.

---

### **Example Calculation**
Suppose:
- $ S_0 = 100 $, $ K = 100 $, $ u = 1.2 $, $ d = 0.8 $, $ r = 0.05 $.
- $ V_T(H) = \max(100 \cdot 1.2 - 100, 0) = 20 $.
- $ V_T(T) = \max(100 \cdot 0.8 - 100, 0) = 0 $.

Calculate risk-neutral probabilities:
$$
\tilde{p} = \frac{(1 + 0.05) - 0.8}{1.2 - 0.8} = \frac{0.25}{0.4} = 0.625, \quad \tilde{q} = 0.375
$$

Expected payoff:
$$
\tilde{p} \cdot 20 + \tilde{q} \cdot 0 = 0.625 \cdot 20 + 0.375 \cdot 0 = 12.5
$$

Present value:
$$
V_0 = \frac{12.5}{1 + 0.05} = \frac{12.5}{1.05} \approx 11.90
$$

Thus, the price of the European call option is **$11.90**.

---

### **Key Takeaways**
1. **Risk-neutral probabilities** are critical for pricing options under the **arbitrage-free assumption**.
2. The formula $ V_0 = \frac{\tilde{p} \cdot V_T(H) + \tilde{q} \cdot V_T(T)}{1 + r} $ generalizes to multi-period models by recursively applying the same logic.
3. This approach assumes **no transaction costs**, **complete markets**, and **log-normal asset price movements** (in more advanced models).

This method is the foundation of **binomial option pricing** and extends to more complex derivatives.


--- Sources ---
  · Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf | domain: finance | score: 0.66
  · Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf | domain: finance | score: 0.66
  · Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf | domain: finance | score: 0.65
  · Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf | domain: finance | score: 0.65
  · A First Course in Stochastic Calculus (Pure and Applied -- Loui

In [75]:
pergunta = "Explain to me how can code a european call option in python or Go?" 

display_figas(pergunta_teste=pergunta, system_prompt=system_prompt)

To code a **European call option** in **Python**, the **Black-Scholes formula** (equation 10.19 in the context) is commonly used. The formula is mathematically expressed as:

$$
C = S_0 \cdot N(d_1) - K \cdot e^{-rT} \cdot N(d_2)
$$

Where:
- $ C $: Price of the European call option.
- $ S_0 $: Current stock price.
- $ K $: Strike price.
- $ r $: Risk-free interest rate.
- $ T $: Time to maturity (in years).
- $ \sigma $: Volatility of the underlying asset.
- $ N(x) $: Cumulative distribution function (CDF) of the standard normal distribution.
- $ d_1 $ and $ d_2 $: Intermediate variables calculated as:
  $$
  d_1 = \frac{\ln\left(\frac{S_0}{K}\right) + \left(r + \frac{\sigma^2}{2}\right)T}{\sigma \sqrt{T}}
  $$
  $$
  d_2 = d_1 - \sigma \sqrt{T}
  $$

### Python Implementation
Here’s how to implement this in Python using `scipy.stats.norm` for $ N(x) $:

```python
import numpy as np
from scipy.stats import norm

def black_scholes_call(S0, K, T, r, sigma):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

# Example usage:
S0 = 100  # Current stock price
K = 100   # Strike price
T = 1     # Time to maturity (1 year)
r = 0.05  # Risk-free rate (5%)
sigma = 0.2  # Volatility (20%)

price = black_scholes_call(S0, K, T, r, sigma)
print(f"European Call Price: {price:.2f}")
```

### Key Notes:
1. **`norm.cdf`**: The CDF of the standard normal distribution is used to compute $ N(d_1) $ and $ N(d_2) $. This is imported from `scipy.stats.norm`.
2. **Parameters**: Ensure inputs like $ S_0, K, T, r, \sigma $ are correctly defined. For example, $ T $ must be in years (e.g., 1 year for a 12-month option).
3. **Monte-Carlo Alternative**: While the Black-Scholes formula is analytical, the context also mentions **Monte-Carlo simulations** (e.g., Example 10.32). However, for a European call, Black-Scholes is more efficient.

### Go Implementation (Conceptual)
Go does not have built-in financial libraries like `scipy`, but you can implement the Black-Scholes formula using standard math libraries. Below is a conceptual outline:

```go
package main

import (
	"fmt"
	"math"
)

func normCDF(x float64) float64 {
	// Approximation of the standard normal CDF (e.g., using the Abramowitz and Stegun approximation)
	return 0.5 * (1 + math.Erfinf(x/math.Sqrt(2)))
}

func blackScholesCall(S0, K, T, r, sigma float64) float64 {
	d1 := (math.Log(S0/K) + (r + 0.5*sigma*sigma)*T) / (sigma * math.Sqrt(T))
	d2 := d1 - sigma*math.Sqrt(T)
	callPrice := S0*normCDF(d1) - K*math.Exp(-r*T)*normCDF(d2)
	return callPrice
}

func main() {
	S0 := 100.0
	K := 100.0
	T := 1.0
	r := 0.05
	sigma := 0.2
	price := blackScholesCall(S0, K, T, r, sigma)
	fmt.Printf("European Call Price: %.2f\n", price)
}
```

### Key Differences:
- **Go**: Requires manual implementation of the CDF (e.g., using `math.Erfinf` for the error function).
- **Python**: Leverages `scipy.stats.norm.cdf` for precision and simplicity.

### Contextual Relevance:
The context emphasizes using the **Black-Scholes formula** (equation 10.19) and Python’s `scipy.stats.norm` for CDF calculations. It also mentions **Monte-Carlo simulations** for exotic options, but for a standard European call, the Black-Scholes approach is preferred.


--- Sources ---
  · Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf | domain: finance | score: 0.68
  · Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf | domain: finance | score: 0.66
  · Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf | domain: finance | score: 0.65
  · Stochastic Calculus for Finance I_ The Binomial Asset -- Steven E_ Shreve -- Springer Finance, 1, 2005 -- Springer US -- 9780387249681 -- 522607e78fc677414b8354ede08772e7 -- Anna’s Archive.pdf | domain: finance | score: 0.65
  · A First Course in Stochastic Calculus (Pure and Applied -- Loui